# Diabeteksen etenemisen ennustaminen lineaarisella regressiolla

## 1. Johdanto

Diabetes on pitkäaikaissairaus, jonka eteneminen vaihtelee potilaiden välillä. Potilaan lähtötilanteessa mitattujen terveystietojen avulla voidaan pyrkiä ennustamaan sairauden myöhempää etenemistä.

Tämän projektin tavoitteena on tutkia, voidaanko diabeteksen etenemistä ennustaa potilaan lähtötilanteessa mitattujen terveystietojen perusteella lineaarisen regression avulla.

**Tutkimuskysymys:**  
Voidaanko diabeteksen etenemistä ennustaa potilaan lähtötilanteessa mitattujen terveystietojen perusteella lineaarisen regression avulla?

Raportissa määritellään ensin koneoppimisongelma ja käytetty aineisto. Tämän jälkeen kuvataan käytetyt piirteet, lineaarinen regressiomalli, häviöfunktio sekä mallin validointi.


In [1]:
# Tarvittavien kirjastojen importti

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

## 2. Ongelman määrittely

Diabeteksen etenemisen ennustaminen muotoillaan ohjatun oppimisen regressio-ongelmaksi, koska ennustettava kohdemuuttuja on jatkuva numeerinen arvo.

Yksi datapiste vastaa yhtä diabetespotilasta. Mallin syötteenä käytetään 10 potilaan lähtötilanteessa mitattua ominaisuuspiirrettä: ikää, sukupuolta, painoindeksiä (BMI), keskimääräistä verenpainetta sekä kuutta veriseerumista mitattua muuttujaa.

Ennustettava kohdemuuttuja (`target`) on numeerinen arvo, joka kuvaa diabeteksen etenemistä vuoden kuluttua lähtötilanteesta.

Projektissa käytetään scikit-learn-kirjaston Diabetes-aineistoa.

**Datan lähde:**  
https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html

## 3. Menetelmät

### 3.1 Aineisto

Aineisto sisältää 442 datapistettä ja 10 potilaan lähtötilanteessa mitattua ominaisuuspiirrettä. Aineisto ladattiin alkuperäisillä skaalaamattomilla arvoilla (`scaled=False`), jotta muuttujien arvot ovat helposti tulkittavissa.

Aineistossa ei ole puuttuvia arvoja, joten puuttuvien arvojen käsittelyä ei tarvita.

In [2]:
# Ladataan Diabetes aineisto DataFramena

diabetes = load_diabetes(as_frame=True, scaled=False)

# Muodostetaan featuret ja kohdemuuttujan sisältävä DataFrame
df = diabetes.frame


# Näytetään ainoston viisi ensimmäistä riviä
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0,151.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0,75.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0,141.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0,206.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0,135.0


In [3]:
# Aineiston koko
df.shape

(442, 11)

In [4]:
# Tarkistetaan puuttuvat arvot
df.isnull().sum()

age       0
sex       0
bmi       0
bp        0
s1        0
s2        0
s3        0
s4        0
s5        0
s6        0
target    0
dtype: int64

### 3.2 Piirteiden valinta

Mallin syötteenä käytetään kaikkia aineiston 10 ominaisuuspiirrettä. Kohdemuuttuja (`target`) erotetaan syötepiirteistä ja sitä käytetään mallin ennustettavana muuttujana.

Kaikki saatavilla olevat ominaisuuspiirteet säilytetään tässä vaiheessa, koska tavoitteena on tutkia niiden yhteistä kykyä ennustaa diabeteksen etenemistä.

In [5]:
# Erotetaan syötepiirteet X ja ennustettava kohdemuuttuja y

X = df.drop(columns="target")
y = df["target"]

print("Piirteiden koko:", X.shape)
print("Kohdemuuttujan koko:", y.shape)

Piirteiden koko: (442, 10)
Kohdemuuttujan koko: (442,)


### 3.3 Lineaarinen regressio

Projektin koneoppimismenetelmäksi valitaan lineaarinen regressio.

Lineaarinen regressio soveltuu tähän ongelmaan, koska tavoitteena on ennustaa jatkuvaa numeerista kohdemuuttujaa useiden syötepiirteiden perusteella. Malli muodostaa lineaarisen yhteyden syötepiirteiden ja ennustettavan kohdemuuttujan välille.

### 3.4 Häviöfunktio

Lineaarisen regression häviöfunktiona käytetään neliövirhettä (squared error). Mallin virhe muodostuu ennustetun arvon ja todellisen arvon välisestä erosta, joka korotetaan toiseen potenssiin.

Mallin suorituskykyä voidaan arvioida keskimääräisellä neliövirheellä (Mean Squared Error, MSE), jossa kaikkien havaintojen neliövirheistä lasketaan keskiarvo.

### 3.5 Mallin validointi

Aineisto jaetaan koulutus-, validaatio- ja testiaineistoihin suhteessa 70 % / 15 % / 15 %. Koulutusaineistossa on 309, validaatioaineistossa 66 ja testiaineistossa 67 datapistettä.

Koulutusaineistoa käytetään mallin opettamiseen, validaatioaineistoa mallin arviointiin kehitysvaiheessa ja testiaineistoa lopullisen mallin suorituskyvyn arviointiin. Testiaineisto pidetään erillään mallin kehityksestä, jotta lopullinen arvio tehdään aineistolla, jota ei ole käytetty mallin kehittämiseen.

In [6]:
# Jaetaan aineisto 70 % koulutusaineistoon ja 30 % väliaikaiseen aineistoon
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Jaetaan jäljelle jäävä aineisto puoliksi validaatio- ja testiaineistoihin
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Koulutusaineisto:", X_train.shape)
print("Validaatioaineisto:", X_val.shape)
print("Testiaineisto:", X_test.shape)

Koulutusaineisto: (309, 10)
Validaatioaineisto: (66, 10)
Testiaineisto: (67, 10)


## 4. Generatiivisen tekoälyn käyttö

Projektissa käytettiin OpenAI ChatGPT:tä projektiohjeiden tulkitsemisen, notebookin rakenteen suunnittelun, tekstin muotoilun ja Python-koodin tarkistamisen tukena. ChatGPT:n rooli oli neuvova.